# Milestone 2 — Task #2: Transformed Outcome Model (Modified Covariate Approach)

## Mathematical Formulation
In standard supervised learning, response models predict the probability of a visit given ad exposure, $P(Y=1 \mid X, T=1)$. In causal incrementality, our objective is to predict the **Individual Treatment Effect (ITE)**:
$$\tau(X) = \mathbb{E}[Y(1) - Y(0) \mid X]$$

Because each user is observed under only one treatment assignment, we apply the transformed outcome identity $Z$:
$$Z = \text{visit} \times \frac{\text{treatment} - p}{p \times (1 - p)}$$

where:
* $\text{treatment} \in \{0, 1\}$ denotes ad exposure.
* $p = P(\text{treatment} = 1)$ is the empirical assignment probability measured on the training split ($\approx 85\%$, not $50\%$).
* Under randomized assignment, $\mathbb{E}[Z \mid X] = \tau(X)$.

## High-Variance Target Dynamics
Because the dataset is skewed towards treatment ($p \approx 0.85$):
* **Treated visitors** ($\text{visit}=1, \text{treatment}=1$):
  $$Z = 1 \times \frac{1 - 0.85}{0.85 \times 0.15} \approx +1.18$$
* **Control visitors** ($\text{visit}=1, \text{treatment}=0$):
  $$Z = 1 \times \frac{0 - 0.85}{0.85 \times 0.15} \approx -6.67$$
* **Non-visitors** ($\text{visit}=0$):
  $$Z = 0$$

Because a control visitor carries over $5.6\times$ the statistical magnitude of a treated visitor, target $Z$ has high variance. We use an L2-regularized linear model (`Ridge`) with standardized features to generate stable uplift predictions $\hat{Z}$.

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load training and validation datasets
df_train = pd.read_csv("data/train.csv.gz")
df_val = pd.read_csv("data/validation.csv.gz")

print(f"Train records: {len(df_train):,}")
print(f"Validation records: {len(df_val):,}")

Train records: 700,000
Validation records: 150,000


In [4]:
feature_cols = [f"f{i}" for i in range(12)]

# 1. Compute empirical treatment probability p on training data only
p = df_train["treatment"].mean()
print(f"Empirical treatment assignment rate (p): {p:.4f}")

# 2. Formulate target variable Z
df_train["Z"] = (
    df_train["visit"]
    * (df_train["treatment"] - p)
    / (p * (1.0 - p))
)

# 3. Verify theoretical target values
z_check = df_train.groupby(["treatment", "visit"])["Z"].agg(["count", "mean"])
print("\nTarget Z Distribution Verification:")
print(z_check)

Empirical treatment assignment rate (p): 0.8500

Target Z Distribution Verification:
                  count      mean
treatment visit                  
0         0      100967  0.000000
          1        4066 -6.664572
1         0      566176  0.000000
          1       28791  1.176536


In [5]:
# Separate features and target
X_train = df_train[feature_cols]
y_train_z = df_train["Z"]

# Build Ridge pipeline with standard scaling
transformed_outcome_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=100.0, random_state=42))
])

# Fit on training split
transformed_outcome_pipeline.fit(X_train, y_train_z)

# Score validation data to generate uplift predictions
X_val = df_val[feature_cols]
df_val["uplift_transformed_outcome"] = transformed_outcome_pipeline.predict(X_val)

print("Validation Uplift Predictions Summary:")
print(df_val["uplift_transformed_outcome"].describe())

Validation Uplift Predictions Summary:
count    150000.000000
mean          0.009756
std           0.031046
min          -0.091033
25%          -0.007634
50%          -0.002007
75%           0.012647
max           0.330672
Name: uplift_transformed_outcome, dtype: float64


### Summary & Findings: Transformed Outcome Model (Task #2)

#### 1. What We Actually Built & Target Behavior
Instead of fitting standard binary classification like our Task 8 response model, we reframed the objective around estimating Individual Treatment Effects (ITE) directly using a single regression pipeline. 

Because our assignment probability is skewed ($p \approx 0.85$ rather than a 50/50 randomized split):
* Non-visitors simply map to $Z = 0.0$.
* Treated visitors evaluate to $Z \approx +1.18$.
* Control visitors evaluate to $Z \approx -6.67$.

This extreme asymmetry makes $Z$ super noisy. A single control visitor exerts over $5.6\times$ the statistical pull of a treated visitor. Using `StandardScaler` paired with `Ridge(alpha=100.0)` helped damp down that high variance so the model didn't blow up on outliers.

#### 2. Validation Predictions Check
Looking at the descriptive stats of `uplift_transformed_outcome` on our unseen validation set:
* **Mean Uplift:** The average predicted score naturally centers around our sample's unadjusted Average Treatment Effect (ATE), sitting at a small positive baseline.
* **Spread & Directionality:** Most predictions cluster tightly around zero with a modest spread across negative and positive tails. Positive scores highlight users whose visit probability is expected to actively jump if treated, while negative scores identify users who are either organic converters that don't need an ad or users where ad intervention could actually backfire.

#### 3. Causal Interpretation vs. Response Modeling
In Task 8, our logistic response model scored users based purely on overall visit probability ($P(Y=1 \mid X, T=1)$), which heavily rewarded high-intent organic visitors regardless of whether the ad actually caused them to visit. The Transformed Outcome approach attempts to solve for true incremental lift ($\tau(X)$) directly. 

While it is mathematically unbiased in expectation, the huge variance in $Z$ means single-model linear regression takes a hit on stability compared to two-model setups. These uplift predictions are now ready to be plugged directly into `src/uplift_eval.py` for Task #3 so we can plot cumulative Qini curves and benchmark against our random baseline and T-Learner.

In [6]:
import os
import numpy as np

os.makedirs("scores", exist_ok=True)
np.save(
    "scores/val_transformed_outcome.npy",
    df_val["uplift_transformed_outcome"].to_numpy(),
)

### Export Validation Scores for Task 3
Save the 150,000 Transformed Outcome validation uplift predictions to `scores/val_transformed_outcome.npy` for model comparison.